In [ ]:
from google.cloud import bigquery
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [ ]:
# Add the name of the project in google console
client = bigquery.Client(project="")

In [52]:
# Creates a CTE then saves the results to query from for all the features  
# We filter out dispositions that aren't ADMITTED or HOME.  We miss intermediate LWBS in ed_pathway which is fine, no clinical information if that happened

query = """
-- ============================================================
-- COHORT IDENTIFICATION: ED VISITS FOR RL PROJECT
-- ADS599 Capstone - MIMIC-IV
-- ============================================================
-- Pathway Labels:
--   ED_DIRECT_ICU          : ED → ICU (no intermediate ward)
--   ED_WARD_ICU            : ED → Ward → ICU (delayed escalation)
--   ED_WARD_DISCHARGE      : ED → Ward → Discharged (no ICU)
--   ED_DISCHARGE_STABLE    : ED → Home, no ICU return or death within 72h
--   ED_DISCHARGE_RETURN_ICU: ED → Home → ED → ICU within 72h
--   ED_DISCHARGE_DIED_72H  : ED → Home → Died within 72h
--
-- Inclusion: disposition IN ('HOME', 'ADMITTED') only.
-- Excluded: TRANSFER (no follow-up data), EXPIRED (died in ED, out of scope),
--           LEFT WITHOUT BEING SEEN / LEFT AGAINST MEDICAL ADVICE / ELOPED
--           (patient-driven departure, not a provider disposition decision).
-- ============================================================

WITH

-- ── 1. All ED visits (scoped to HOME or ADMITTED dispositions) ────────────────
ed_base AS (
  SELECT
    e.subject_id,
    e.stay_id         AS ed_stay_id,
    e.hadm_id,
    e.intime          AS ed_intime,
    e.outtime         AS ed_outtime,
    e.disposition,
    e.race,                -- from edstays: always populated (admissions.race is NULL for non-admitted)
    e.arrival_transport    -- WALK IN / AMBULANCE / HELICOPTER / UNKNOWN / OTHER
  FROM `physionet-data.mimiciv_ed.edstays` e
  WHERE e.disposition IN ('HOME', 'ADMITTED')
),

-- ── 2. ICU care unit names (MIMIC-IV transfers table values) ─────────────────
icu_units AS (
  SELECT careunit FROM UNNEST([
    'Medical Intensive Care Unit (MICU)',
    'Surgical Intensive Care Unit (SICU)',
    'Cardiac Vascular Intensive Care Unit (CVICU)',
    'Medical/Surgical Intensive Care Unit (MICU/SICU)',
    'Coronary Care Unit (CCU)',
    'Trauma SICU (TSICU)',
    'Neuro Surgical Intensive Care Unit (Neuro SICU)',
    'Neuro Intermediate'
  ]) AS careunit
),

-- ── 3. Transfer events after the ED (excluding ED unit itself, excluding discharge events) ──
transfers_flagged AS (
  SELECT
    t.subject_id,
    t.hadm_id,
    t.careunit,
    t.intime,
    CASE WHEN u.careunit IS NOT NULL THEN 1 ELSE 0 END AS is_icu
  FROM `physionet-data.mimiciv_3_1_hosp.transfers` t
  LEFT JOIN icu_units u USING (careunit)
  WHERE t.eventtype != 'discharge'
    AND (t.careunit NOT LIKE '%Emergency%' OR t.careunit IS NULL)
),

-- ── 4. Per admission: ever reach ICU, and when? ───────────────────────────────
hadm_icu_summary AS (
  SELECT
    hadm_id,
    MAX(is_icu)                               AS ever_icu,
    MIN(CASE WHEN is_icu = 1 THEN intime END) AS first_icu_intime
  FROM transfers_flagged
  GROUP BY hadm_id
),

-- ── 5. Per admission: what was the FIRST non-ED care unit? ───────────────────
hadm_first_unit AS (
  SELECT
    tf.hadm_id,
    tf.careunit AS first_careunit,
    tf.is_icu   AS first_is_icu
  FROM transfers_flagged tf
  INNER JOIN (
    SELECT hadm_id, MIN(intime) AS min_intime
    FROM transfers_flagged
    GROUP BY hadm_id
  ) fm ON tf.hadm_id = fm.hadm_id AND tf.intime = fm.min_intime
),

-- ── 6. Primary pathway per ED visit ──────────────────────────────────────────
ed_pathway AS (
  SELECT
    eb.subject_id,
    eb.ed_stay_id,
    eb.hadm_id,
    eb.ed_intime,
    eb.ed_outtime,
    eb.disposition,
    eb.race,
    eb.arrival_transport,
    his.ever_icu,
    his.first_icu_intime,
    hfu.first_careunit,
    CASE
      WHEN eb.hadm_id IS NOT NULL AND his.ever_icu = 1 AND hfu.first_is_icu = 1
        THEN 'ED_DIRECT_ICU'
      WHEN eb.hadm_id IS NOT NULL AND his.ever_icu = 1 AND COALESCE(hfu.first_is_icu, 0) = 0
        THEN 'ED_WARD_ICU'
      WHEN eb.hadm_id IS NOT NULL AND COALESCE(his.ever_icu, 0) = 0
        THEN 'ED_WARD_DISCHARGE'
      WHEN eb.hadm_id IS NULL
        THEN 'ED_DISCHARGE'
      ELSE 'UNKNOWN'
    END AS base_pathway
  FROM ed_base eb
  LEFT JOIN hadm_icu_summary his ON eb.hadm_id = his.hadm_id
  LEFT JOIN hadm_first_unit  hfu ON eb.hadm_id = hfu.hadm_id
),

-- ── 7. Bounced back: discharge → return ED → ICU within 72h ─────────────────
bounced_icu AS (
  SELECT DISTINCT ep1.ed_stay_id AS original_ed_stay_id
  FROM ed_pathway ep1
  INNER JOIN ed_pathway ep2
    ON  ep1.subject_id = ep2.subject_id
    AND ep2.ed_intime  > ep1.ed_outtime
    AND TIMESTAMP_DIFF(ep2.ed_intime, ep1.ed_outtime, HOUR) <= 72
  WHERE ep1.base_pathway = 'ED_DISCHARGE'
    AND ep2.base_pathway IN ('ED_DIRECT_ICU', 'ED_WARD_ICU')
),

-- ── 8. Died within 72h of ED discharge ───────────────────────────────────────
died_72h AS (
  SELECT ep.ed_stay_id
  FROM ed_pathway ep
  INNER JOIN `physionet-data.mimiciv_3_1_hosp.patients` p
    ON ep.subject_id = p.subject_id
  WHERE ep.base_pathway = 'ED_DISCHARGE'
    AND p.dod IS NOT NULL
    AND DATE_DIFF(p.dod, DATE(ep.ed_outtime), DAY) BETWEEN 0 AND 3
)

-- ── 9. Final labeled cohort ───────────────────────────────────────────────────
SELECT
  ep.subject_id,
  ep.ed_stay_id,
  ep.hadm_id,
  ep.ed_intime,
  ep.ed_outtime,
  ep.disposition,
  ep.race,               -- from edstays (always populated)
  ep.arrival_transport,
  ep.first_careunit,
  ep.first_icu_intime,
  ep.base_pathway,
  -- Granular cohort label
  CASE
    WHEN ep.base_pathway IN ('ED_DIRECT_ICU', 'ED_WARD_ICU', 'ED_WARD_DISCHARGE')
      THEN ep.base_pathway
    WHEN ep.base_pathway = 'ED_DISCHARGE' AND bi.original_ed_stay_id IS NOT NULL
      THEN 'ED_DISCHARGE_RETURN_ICU'
    WHEN ep.base_pathway = 'ED_DISCHARGE' AND d72.ed_stay_id IS NOT NULL
      THEN 'ED_DISCHARGE_DIED_72H'
    WHEN ep.base_pathway = 'ED_DISCHARGE'
      THEN 'ED_DISCHARGE_STABLE'
    ELSE 'UNKNOWN'
  END AS cohort_label,
  -- Demographics
  p.gender,
  p.anchor_age,
  p.anchor_year,
  -- Approximate age at ED visit (anchor_age + years since anchor_year)
  p.anchor_age + (EXTRACT(YEAR FROM ep.ed_intime) - p.anchor_year) AS age_at_visit,
  p.dod,
  -- Admission info (NULL for pure ED discharges)
  a.admittime,
  a.dischtime,
  a.admission_type,
  a.discharge_location,
  a.insurance,
  a.language,
  a.marital_status
FROM ed_pathway ep
INNER JOIN `physionet-data.mimiciv_3_1_hosp.patients` p
  ON ep.subject_id = p.subject_id
LEFT JOIN `physionet-data.mimiciv_3_1_hosp.admissions` a
  ON ep.hadm_id = a.hadm_id
LEFT JOIN bounced_icu bi  ON ep.ed_stay_id = bi.original_ed_stay_id
LEFT JOIN died_72h    d72 ON ep.ed_stay_id = d72.ed_stay_id
ORDER BY ep.subject_id, ep.ed_intime
;
"""

In [53]:
# Save cohort query results to a BigQuery table so feature queries can reference it
# without repeating the full CTE chain in every query.
from google.api_core.exceptions import NotFound

dataset_ref = client.dataset("rl_project", project="ads-599-final")
try:
    client.get_dataset(dataset_ref)
    print("Dataset rl_project already exists")
except NotFound:
    from google.cloud.bigquery import Dataset
    ds = Dataset(dataset_ref)
    ds.location = "US"
    client.create_dataset(ds)
    print("Created dataset: ads-599-final.rl_project")

job_config = bigquery.QueryJobConfig(
    destination="ads-599-final.rl_project.cohort_base",
    write_disposition="WRITE_TRUNCATE"
)
client.query(query, job_config=job_config).result()
print("Cohort saved to BigQuery: ads-599-final.rl_project.cohort_base")

df_cohort = client.query("SELECT * FROM `ads-599-final.rl_project.cohort_base`").to_dataframe()
print(f"Shape: {df_cohort.shape}")
df_cohort.head()

Dataset rl_project already exists
Cohort saved to BigQuery: ads-599-final.rl_project.cohort_base
Shape: (399573, 24)


,subject_id,ed_stay_id,hadm_id,ed_intime,ed_outtime,disposition,race,arrival_transport,first_careunit,first_icu_intime,...,anchor_year,age_at_visit,dod,admittime,dischtime,admission_type,discharge_location,insurance,language,marital_status
0,10002013,34931809,21763296,2165-11-22 20:54:00,2165-11-23 08:20:42,ADMITTED,WHITE,WALK IN,Medical Intensive Care Unit (MICU),2165-11-23 08:20:42,...,2156,62,NaT,2165-11-23 08:19:00,2165-11-26 15:40:00,DIRECT EMER.,HOME HEALTH CARE,Medicaid,English,SINGLE
1,10014354,38849383,27562275,2148-07-17 20:36:00,2148-07-18 04:46:00,ADMITTED,WHITE,AMBULANCE,Cardiology Surgery Intermediate,NaT,...,2146,62,2153-04-07,2148-07-18 02:31:00,2148-07-20 17:47:00,DIRECT EMER.,HOME HEALTH CARE,Medicare,English,SINGLE
2,10018862,35211473,22949345,2149-05-01 17:05:00,2149-05-01 22:22:00,ADMITTED,WHITE,WALK IN,Transplant,NaT,...,2148,57,NaT,2149-05-01 21:43:00,2149-05-05 18:20:00,DIRECT EMER.,HOME,Medicaid,English,MARRIED
3,10019003,38020791,26529390,2155-05-17 21:03:00,2155-05-18 00:03:15,ADMITTED,WHITE,WALK IN,Hematology/Oncology Intermediate,NaT,...,2148,72,2155-12-03,2155-05-17 00:02:00,2155-05-19 18:27:00,DIRECT EMER.,HOME,Medicare,English,WIDOWED
4,10021395,31017572,21372240,2132-07-30 18:58:00,2132-07-31 04:56:00,ADMITTED,WHITE - OTHER EUROPEAN,WALK IN,Medicine,NaT,...,2128,90,NaT,2132-07-31 03:28:00,2132-07-31 15:09:00,DIRECT EMER.,HOME,Medicare,English,WIDOWED


In [54]:
df_cohort.to_csv("cohort_base.csv", index=False)
print("Saved cohort_base.csv")

# Verification
print("\n--- Cohort Label Distribution ---")
print(df_cohort['cohort_label'].value_counts())
print(f"\nNull race count: {df_cohort['race'].isna().sum()}  (should be 0)")
print(f"Total ED visits: {len(df_cohort):,}")

Saved cohort_base.csv

--- Cohort Label Distribution ---
cohort_label
ED_DISCHARGE_STABLE        205193
ED_WARD_DISCHARGE          162478
ED_DIRECT_ICU               23088
ED_WARD_ICU                  8533
ED_DISCHARGE_RETURN_ICU       224
ED_DISCHARGE_DIED_72H          57
Name: count, dtype: int64

Null race count: 0  (should be 0)
Total ED visits: 399,573


## Feature Extraction Queries

All queries below use `ads-599-final.rl_project.cohort_base` as the cohort filter.
Each produces one output file. Static tables (triage, medrecon) have 1 row per ED visit;
sequential tables (vitals, labs, ECG, pyxis, radiology) have N rows per visit with timestamps.

In [55]:
# --- Query 1: Triage (static, 1 row per ED visit) ---
# Chief complaint, ESI acuity level, and initial vitals recorded at triage.
# One-to-one with edstays, so row count should match cohort size.

query_triage = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  t.stay_id   AS ed_stay_id,
  t.subject_id,
  t.temperature,
  t.heartrate,
  t.resprate,
  t.o2sat,
  t.sbp,
  t.dbp,
  t.pain,
  t.acuity,
  t.chiefcomplaint
FROM `physionet-data.mimiciv_ed.triage` t
INNER JOIN cohort_subjects cs ON t.stay_id = cs.ed_stay_id
ORDER BY t.stay_id
"""

print("Running Query 1: Triage...")
df_triage = client.query(query_triage).to_dataframe()
print(f"Shape: {df_triage.shape}")
df_triage.to_csv("triage.csv", index=False)
print(f"Saved triage.csv")
print(f"Triage rows vs cohort rows: {len(df_triage):,} vs {len(df_cohort):,}  (should be ~equal)")
df_triage.head()

Running Query 1: Triage...
Shape: (399573, 11)
Saved triage.csv
Triage rows vs cohort rows: 399,573 vs 399,573  (should be ~equal)


,ed_stay_id,subject_id,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint
0,30000012,11714491,98.800000000,96.000000000,18.000000000,93.000000000,160.000000000,54.000000000,0,2.000000000,CHANGE IN MENTAL STATUS
1,30000038,13821532,97.100000000,54.000000000,18.000000000,95.000000000,143.000000000,73.000000000,0,3.000000000,Cough
2,30000039,13340997,98.600000000,85.000000000,16.000000000,98.000000000,189.000000000,96.000000000,0,3.000000000,s/p Fall
3,30000055,19848164,99.400000000,85.000000000,16.000000000,100.000000000,None,None,0,3.000000000,L Ear pain
4,30000094,19862552,98.100000000,60.000000000,18.000000000,94.000000000,120.000000000,95.000000000,2,2.000000000,N


In [56]:
# --- Query 2: Medrecon (static — pre-arrival medications) ---
# Medications the patient was already taking before the ED visit.
# Treated as static/baseline state (like demographics), not time-varying.
# Multiple rows per visit: one per drug per ETC therapeutic class group.
# etcdescription groups drugs by class (e.g., 'Beta Blockers', 'ACE Inhibitors').

query_medrecon = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  m.stay_id       AS ed_stay_id,
  m.subject_id,
  m.charttime,
  m.name,
  m.gsn,
  m.ndc,
  m.etccode,
  m.etcdescription
FROM `physionet-data.mimiciv_ed.medrecon` m
INNER JOIN cohort_subjects cs ON m.stay_id = cs.ed_stay_id
ORDER BY m.stay_id, m.name
"""

print("Running Query 2: Medrecon...")
df_medrecon = client.query(query_medrecon).to_dataframe()
print(f"Shape: {df_medrecon.shape}")
df_medrecon.to_csv("medrecon.csv", index=False)
print(f"Saved medrecon.csv")
print(f"Unique ED visits with any medrecon: {df_medrecon['ed_stay_id'].nunique():,}")
df_medrecon.head()

Running Query 2: Medrecon...
Shape: (2885116, 8)
Saved medrecon.csv
Unique ED visits with any medrecon: 296,089


,ed_stay_id,subject_id,charttime,name,gsn,ndc,etccode,etcdescription
0,30000012,11714491,2126-02-14 22:21:00,furosemide,008209,10544058430,00000250,Diuretic - Loop
1,30000012,11714491,2126-02-15 00:16:00,gabapentin,021413,10135064401,00006030,Anticonvulsant - GABA Analogs
2,30000012,11714491,2126-02-14 22:21:00,gabapentin,021413,10135064401,00006030,Anticonvulsant - GABA Analogs
3,30000012,11714491,2126-02-14 22:22:00,lactulose,068217,17856137801,00000409,Laxative - Saline and Osmotic
4,30000012,11714491,2126-02-14 22:22:00,magnesium,001419,11694083501,00000740,Minerals and Electrolytes - Magnesium


In [57]:
# --- Query 3: Serial Vital Signs (time-varying state) ---
# Periodic vital sign measurements taken throughout the ED stay.
# Multiple rows per visit, ordered by charttime — tracks patient deterioration/improvement.
# rhythm is cardiac rhythm (e.g., 'Sinus Rhythm', 'Atrial Fibrillation') — only in vitalsign, not triage.

query_vitals = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  v.stay_id   AS ed_stay_id,
  v.subject_id,
  v.charttime,
  v.temperature,
  v.heartrate,
  v.resprate,
  v.o2sat,
  v.sbp,
  v.dbp,
  v.rhythm,
  v.pain
FROM `physionet-data.mimiciv_ed.vitalsign` v
INNER JOIN cohort_subjects cs ON v.stay_id = cs.ed_stay_id
ORDER BY v.stay_id, v.charttime
"""

print("Running Query 3: Vital signs...")
df_vitals = client.query(query_vitals).to_dataframe()
print(f"Shape: {df_vitals.shape}")
df_vitals.to_csv("vitals.csv", index=False)
print(f"Saved vitals.csv")
print(f"Unique ED visits with vitals: {df_vitals['ed_stay_id'].nunique():,}")
print(f"Avg vital readings per visit: {len(df_vitals) / df_vitals['ed_stay_id'].nunique():.1f}")
df_vitals.head()

Running Query 3: Vital signs...
Shape: (1496038, 11)
Saved vitals.csv
Unique ED visits with vitals: 387,600
Avg vital readings per visit: 3.9


,ed_stay_id,subject_id,charttime,temperature,heartrate,resprate,o2sat,sbp,dbp,rhythm,pain
0,30000012,11714491,2126-02-14 20:22:00,98.800000000,96.000000000,18.000000000,93.000000000,160,54,None,0
1,30000012,11714491,2126-02-14 23:43:00,None,80.000000000,13.000000000,99.000000000,112,44,None,0
2,30000012,11714491,2126-02-15 00:50:00,98.600000000,88.000000000,16.000000000,100.000000000,135,51,None,0
3,30000038,13821532,2152-12-07 16:38:00,97.100000000,54.000000000,18.000000000,95.000000000,143,73,None,0
4,30000038,13821532,2152-12-07 19:17:00,None,78.000000000,20.000000000,96.000000000,141,61,None,0


In [58]:
# --- Query 4: Lab Results — ED Visit Window (time-varying state) ---
# Lab tests ordered and resulted during the ED stay.
# labevents has no ed_stay_id — must join via subject_id + charttime window.
# Filtered to common ED labs to avoid pulling 100M+ rows from the full labevents table.
# NOTE: This query may take several minutes due to labevents table size.

query_labs = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id, ed_intime, ed_outtime
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  cs.ed_stay_id,
  le.subject_id,
  le.itemid,
  dl.label,
  dl.fluid,
  le.charttime,
  le.valuenum,
  le.valueuom,
  le.flag,
  le.ref_range_lower,
  le.ref_range_upper
FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
INNER JOIN cohort_subjects cs
  ON le.subject_id = cs.subject_id
  AND le.charttime >= cs.ed_intime
  AND le.charttime <= cs.ed_outtime
INNER JOIN `physionet-data.mimiciv_3_1_hosp.d_labitems` dl
  ON le.itemid = dl.itemid
WHERE dl.label IN (
  -- CBC
  'White Blood Cells', 'Red Blood Cells', 'Hemoglobin', 'Hematocrit',
  'MCV', 'Platelet Count', 'Neutrophils', 'Lymphocytes',
  -- BMP/CMP
  'Sodium', 'Potassium', 'Chloride', 'Bicarbonate', 'Anion Gap',
  'Urea Nitrogen', 'Creatinine', 'Glucose', 'Calcium, Total',
  -- Cardiac markers
  'Troponin T', 'Troponin I', 'NT-proBNP', 'BNP', 'CK-MB',
  -- Infection / metabolic
  'Lactate', 'Procalcitonin', 'C-Reactive Protein',
  'ALT', 'AST', 'Alkaline Phosphatase', 'Bilirubin, Total', 'Albumin',
  'Lipase', 'Amylase',
  -- Coagulation
  'PT', 'PTT', 'INR(PT)', 'D-Dimer', 'Fibrinogen, Functional',
  -- Thyroid
  'TSH',
  -- ABG
  'pH', 'pO2', 'pCO2', 'Base Excess', 'Oxygen Saturation'
)
ORDER BY cs.ed_stay_id, le.charttime
"""

print("Running Query 4: Lab results (this may take several minutes)...")
df_labs = client.query(query_labs).to_dataframe()
print(f"Shape: {df_labs.shape}")
df_labs.to_csv("labs.csv", index=False)
print(f"Saved labs.csv")
print(f"Unique ED visits with labs: {df_labs['ed_stay_id'].nunique():,}")
print(f"Lab types found:\n{df_labs['label'].value_counts().head(20)}")
df_labs.head()

Running Query 4: Lab results (this may take several minutes)...
Shape: (6405081, 11)
Saved labs.csv
Unique ED visits with labs: 303,416
Lab types found:
label
Glucose              488301
Hemoglobin           305041
Creatinine           301995
Urea Nitrogen        301079
Hematocrit           299924
White Blood Cells    298001
Platelet Count       297875
Red Blood Cells      297570
MCV                  297570
Potassium            294824
Sodium               294630
Chloride             294515
Bicarbonate          293693
Anion Gap            293257
Lymphocytes          287018
Neutrophils          283891
pH                   208111
INR(PT)              137156
PT                   137156
PTT                  136342
Name: count, dtype: int64


,ed_stay_id,subject_id,itemid,label,fluid,charttime,valuenum,valueuom,flag,ref_range_lower,ref_range_upper
0,30000012,11714491,51250,MCV,Blood,2126-02-14 22:15:00,99.0,fL,abnormal,82.0,98.0
1,30000012,11714491,50885,"Bilirubin, Total",Blood,2126-02-14 22:15:00,8.1,mg/dL,abnormal,0.0,1.5
2,30000012,11714491,51265,Platelet Count,Blood,2126-02-14 22:15:00,43.0,K/uL,abnormal,150.0,400.0
3,30000012,11714491,51301,White Blood Cells,Blood,2126-02-14 22:15:00,4.5,K/uL,None,4.0,10.0
4,30000012,11714491,51222,Hemoglobin,Blood,2126-02-14 22:15:00,8.9,g/dL,abnormal,11.2,15.7


In [59]:
# --- Query 5: ECG Machine Measurements — Near ED Visit (time-varying state) ---
# Automated ECG measurements (RR interval, QRS axis, etc.) and machine-generated report text.
# Join via subject_id + ecg_time within ±2 hours of ED window.
#
# ⚠️  KNOWN LIMITATION: ecg_time is from the ECG machine's internal clock, which is frequently
# out of sync with hospital systems. The ±2 hour buffer is a pragmatic tolerance.
# ~25% of cohort patients are expected to have at least one ECG near their ED visit.

query_ecg = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id, ed_intime, ed_outtime
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  cs.ed_stay_id,
  rl.subject_id,
  rl.study_id,
  mm.ecg_time,
  mm.rr_interval,
  mm.p_onset,
  mm.p_end,
  mm.qrs_onset,
  mm.qrs_end,
  mm.t_end,
  mm.p_axis,
  mm.qrs_axis,
  mm.t_axis,
  -- Concatenate machine report text segments (up to report_5; extend if needed)
  TRIM(CONCAT(
    COALESCE(mm.report_0, ''), ' ', COALESCE(mm.report_1, ''), ' ',
    COALESCE(mm.report_2, ''), ' ', COALESCE(mm.report_3, ''), ' ',
    COALESCE(mm.report_4, ''), ' ', COALESCE(mm.report_5, '')
  )) AS machine_report
FROM `physionet-data.mimiciv_ecg.record_list` rl
INNER JOIN `physionet-data.mimiciv_ecg.machine_measurements` mm
  ON rl.study_id = mm.study_id
INNER JOIN cohort_subjects cs
  ON rl.subject_id = cs.subject_id
  AND mm.ecg_time >= TIMESTAMP_SUB(cs.ed_intime, INTERVAL 2 HOUR)
  AND mm.ecg_time <= TIMESTAMP_ADD(cs.ed_outtime, INTERVAL 2 HOUR)
ORDER BY cs.ed_stay_id, mm.ecg_time
"""

print("Running Query 5: ECG machine measurements...")
df_ecg = client.query(query_ecg).to_dataframe()
print(f"Shape: {df_ecg.shape}")
df_ecg.to_csv("ecg.csv", index=False)
print(f"Saved ecg.csv")
ecg_coverage = df_ecg['ed_stay_id'].nunique() / len(df_cohort) * 100
print(f"Cohort ED visits with ≥1 ECG: {df_ecg['ed_stay_id'].nunique():,} ({ecg_coverage:.1f}%)  (expect ~25%)")
df_ecg.head()

Running Query 5: ECG machine measurements...
Shape: (185757, 14)
Saved ecg.csv
Cohort ED visits with ≥1 ECG: 144,152 (36.1%)  (expect ~25%)


,ed_stay_id,subject_id,study_id,ecg_time,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,machine_report
0,30000012,11714491,46038037,2126-02-14 20:36:00,698,348,29999,506,600,903,72,-33,46,Sinus rhythm Probable left ventricular hypertr...
1,30000039,13340997,45216599,2165-10-06 12:09:00,789,40,168,228,324,632,70,24,33,Sinus rhythm with PVC(s) Borderline ECG
2,30000094,19862552,43682756,2183-09-04 15:00:00,923,40,150,194,282,624,-1,-2,-3,Sinus rhythm Poor R wave progression - probabl...
3,30000094,19862552,48171271,2183-09-04 16:18:00,455,345,29999,499,582,823,32767,-13,2,Atrial fibrillation
4,30000094,19862552,41766637,2183-09-04 16:19:00,488,369,29999,499,586,833,32767,-14,2,Atrial fibrillation


In [60]:
# --- Query 6: Pyxis Dispenses — Medications Given in ED (action records) ---
# Medications dispensed from the Pyxis automated cabinet during the ED stay.
# These represent clinical actions taken by the care team.
# Note: Dispensing from Pyxis does not guarantee administration; nurses may return medications.
#
# Deduplication: A single dispensation event (med_rn) can appear as multiple rows when
# a drug maps to multiple GSN ontology groups (gsn_rn > 1). SELECT DISTINCT on the key
# columns collapses these duplicates while preserving truly separate dispense events.

query_pyxis = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT DISTINCT
  p.stay_id   AS ed_stay_id,
  p.subject_id,
  p.charttime,
  p.med_rn,
  p.name,
  p.gsn
FROM `physionet-data.mimiciv_ed.pyxis` p
INNER JOIN cohort_subjects cs ON p.stay_id = cs.ed_stay_id
ORDER BY p.stay_id, p.charttime, p.med_rn
"""

print("Running Query 6: Pyxis dispenses...")
df_pyxis = client.query(query_pyxis).to_dataframe()
print(f"Shape: {df_pyxis.shape}")
df_pyxis.to_csv("pyxis.csv", index=False)
print(f"Saved pyxis.csv")
print(f"Unique ED visits with Pyxis dispenses: {df_pyxis['ed_stay_id'].nunique():,}")
print(f"Top dispensed medications:\n{df_pyxis['name'].value_counts().head(10)}")
df_pyxis.head()

Running Query 6: Pyxis dispenses...
Shape: (1508744, 6)
Saved pyxis.csv
Unique ED visits with Pyxis dispenses: 292,135
Top dispensed medications:
name
Ondansetron                         82404
Acetaminophen                       43968
Lorazepam                           40777
Ondansetron 4mg/2mL 2mL VIAL        40068
Morphine Sulfate                    38636
HYDROmorphone (Dilaudid)            36229
Morphine                            28685
Vancomycin                          26766
Acetaminophen 500mg TAB             25723
Morphine Sulfat 4mg/1mL 1mL VIAL    25358
Name: count, dtype: int64


,ed_stay_id,subject_id,charttime,med_rn,name,gsn
0,30000012,11714491,2126-02-15 00:44:00,1,Spironolactone,006817
1,30000012,11714491,2126-02-15 00:44:00,2,Gabapentin,021413
2,30000012,11714491,2126-02-15 01:22:00,3,CefTRIAXone (Mini Bag Plus),None
3,30000038,13821532,2152-12-07 18:31:00,1,CefTRIAXone 1gm/100mL 100mL Bag,None
4,30000038,13821532,2152-12-07 18:32:00,2,Azithromyc 500mg/250mL 250mL BAG,None


In [ ]:
# --- Query 7: Radiology Reports — ED Visit Window (action records + state context) ---
# Imaging studies ordered and resulted during the ED stay.
# hadm_id in note.radiology is NULL for non-admitted patients — must use subject_id + time window.
# Filter note_type = 'RR' for primary reports only (exclude 'AR' addenda).
# exam_name and cpt_code pulled from radiology_detail to identify imaging modality.
#
# ⚠️  COVERAGE NOTE: Radiology reports are only reliably present for years 2011–2016.
# For visits outside this range, absence of a record does NOT mean no imaging was ordered.
# The in_radiology_coverage_window flag marks which records fall in the reliable window.

query_radiology = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id, ed_intime, ed_outtime
  FROM `ads-599-final.rl_project.cohort_base`
),
radiology_with_detail AS (
  SELECT
    r.note_id,
    r.subject_id,
    r.charttime,
    r.text AS report_text,
    r.note_type,
    MAX(CASE WHEN rd.field_name = 'exam_name' THEN rd.field_value END) AS exam_name,
    MAX(CASE WHEN rd.field_name = 'cpt_code'  THEN rd.field_value END) AS cpt_code
  FROM `physionet-data.mimiciv_note.radiology` r
  LEFT JOIN `physionet-data.mimiciv_note.radiology_detail` rd
    ON r.note_id = rd.note_id
    AND rd.field_name IN ('exam_name', 'cpt_code')
  WHERE r.note_type = 'RR'
  GROUP BY r.note_id, r.subject_id, r.charttime, r.text, r.note_type
)
SELECT
  cs.ed_stay_id,
  rwd.subject_id,
  rwd.note_id,
  rwd.charttime,
  rwd.exam_name,
  rwd.cpt_code,
  rwd.report_text,
  CASE
    WHEN EXTRACT(YEAR FROM rwd.charttime) BETWEEN 2011 AND 2016
    THEN TRUE ELSE FALSE
  END AS in_radiology_coverage_window
FROM radiology_with_detail rwd
INNER JOIN cohort_subjects cs
  ON rwd.subject_id = cs.subject_id
  AND rwd.charttime >= cs.ed_intime
  AND rwd.charttime <= cs.ed_outtime
ORDER BY cs.ed_stay_id, rwd.charttime
"""

print("Running Query 7: Radiology reports...")
df_radiology = client.query(query_radiology).to_dataframe()
print(f"Shape: {df_radiology.shape}")
df_radiology.to_csv("radiology.csv", index=False)
print(f"Saved radiology.csv")
print(f"Unique ED visits with imaging: {df_radiology['ed_stay_id'].nunique():,}")
print(f"\nRadiology coverage window distribution:")
print(df_radiology['in_radiology_coverage_window'].value_counts())
print(f"\nTop exam types:\n{df_radiology['exam_name'].value_counts().head(10)}")
df_radiology.head()

In [5]:
# --- Query 8: Clinical Services (context for admitted patients) ---
# Which clinical service took responsibility for the patient after the ED.
# Only populated for admitted patients (ED_DIRECT_ICU, ED_WARD_ICU, ED_WARD_DISCHARGE).
# Takes the FIRST service assignment per hospital admission.

query_services = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id, hadm_id
  FROM `ads-599-final.rl_project.cohort_base`
  WHERE hadm_id IS NOT NULL
),
first_service AS (
  SELECT
    hadm_id,
    MIN(transfertime) AS first_service_time
  FROM `physionet-data.mimiciv_3_1_hosp.services`
  GROUP BY hadm_id
)
SELECT
  cs.ed_stay_id,
  s.subject_id,
  s.hadm_id,
  s.transfertime,
  s.curr_service,
  s.prev_service
FROM `physionet-data.mimiciv_3_1_hosp.services` s
INNER JOIN cohort_subjects cs ON s.hadm_id = cs.hadm_id
INNER JOIN first_service fs
  ON s.hadm_id = fs.hadm_id
  AND s.transfertime = fs.first_service_time
ORDER BY cs.ed_stay_id
"""

print("Running Query 8: Clinical services...")
df_services = client.query(query_services).to_dataframe()
print(f"Shape: {df_services.shape}")
df_services.to_csv("services.csv", index=False)
print(f"Saved services.csv")
print(f"\nService distribution:\n{df_services['curr_service'].value_counts().head(15)}")
df_services.head()

Running Query 8: Clinical services...
Shape: (194099, 6)
Saved services.csv

Service distribution:
curr_service
MED      128621
CMED      14090
NMED      11330
SURG      11212
OMED       8982
ORTHO      5532
NSURG      4296
TRAUM      3924
VSURG      1778
GYN        1100
GU          848
TSURG       574
CSURG       541
OBS         503
PSURG       447
Name: count, dtype: int64


,ed_stay_id,subject_id,hadm_id,transfertime,curr_service,prev_service
0,30000012,11714491,21562392,2126-02-15 00:16:37,MED,NaN
1,30000038,13821532,26255538,2152-12-07 19:55:00,OMED,NaN
2,30000039,13340997,23100190,2165-10-06 17:50:27,MED,NaN
3,30000177,17937834,23831044,2143-12-28 01:42:53,MED,NaN
4,30000204,11615015,25540031,2132-10-10 11:41:17,MED,NaN


In [6]:
# --- Query 9: ICU Stays (outcome — time-to-ICU computation) ---
# ICU stay details for patients who were admitted to the ICU during their hospital admission.
# minutes_ed_to_icu is the key outcome variable for the RL reward function.
# NOTE: icu.icustays.stay_id is the ICU stay ID — completely different from the ED stay_id.
#       The ICU stay_id is aliased as icu_stay_id here to avoid confusion.

query_icu = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id, hadm_id, ed_intime
  FROM `ads-599-final.rl_project.cohort_base`
  WHERE hadm_id IS NOT NULL
)
SELECT
  cs.ed_stay_id,
  i.subject_id,
  i.hadm_id,
  i.stay_id                                                      AS icu_stay_id,
  i.intime                                                       AS icu_intime,
  i.outtime                                                      AS icu_outtime,
  i.los,
  i.first_careunit,
  i.last_careunit,
  TIMESTAMP_DIFF(i.intime, cs.ed_intime, MINUTE)                AS minutes_ed_to_icu
FROM `physionet-data.mimiciv_3_1_icu.icustays` i
INNER JOIN cohort_subjects cs ON i.hadm_id = cs.hadm_id
ORDER BY cs.ed_stay_id, i.intime
"""

print("Running Query 9: ICU stays...")
df_icu = client.query(query_icu).to_dataframe()
print(f"Shape: {df_icu.shape}")
df_icu.to_csv("icu_stays.csv", index=False)
print(f"Saved icu_stays.csv")
print(f"Unique ED visits with ICU stay: {df_icu['ed_stay_id'].nunique():,}")
print(f"\nminutes_ed_to_icu summary (should all be positive):")
print(df_icu['minutes_ed_to_icu'].describe())
df_icu.head()

Running Query 9: ICU stays...
Shape: (34466, 10)
Saved icu_stays.csv
Unique ED visits with ICU stay: 31,229

minutes_ed_to_icu summary (should all be positive):
count        34466.0
mean      2893.72367
std      8623.184622
min              8.0
25%            224.0
50%            373.0
75%           1129.0
max         289027.0
Name: minutes_ed_to_icu, dtype: Float64


,ed_stay_id,subject_id,hadm_id,icu_stay_id,icu_intime,icu_outtime,los,first_careunit,last_careunit,minutes_ed_to_icu
0,30000368,18563034,29198602,34044260,2162-10-26 19:06:45,2162-10-28 20:27:02,2.055752,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),179
1,30000785,17857442,29149108,37813117,2121-09-23 10:21:14,2121-09-26 16:23:15,3.251400,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),17316
2,30001785,16061348,28146972,39134325,2151-09-02 17:05:54,2151-09-05 20:07:05,3.125822,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),142
3,30001975,17848638,20077376,32823838,2159-11-12 09:43:12,2159-11-14 14:27:50,2.197662,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),4930
4,30002088,17132849,22584119,30983000,2157-12-16 05:05:19,2157-12-20 14:27:40,4.390521,Coronary Care Unit (CCU),Coronary Care Unit (CCU),59
